<a href="https://colab.research.google.com/github/Sno-7178/Cross-catalyst-estimation-of-Activation-Energy-for-reactions-of-Hydrodeoxygenation-of-propanoic-acid/blob/main/Descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit -qqq

In [ ]:
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, GraphDescriptors, rdMolDescriptors

**Computing the molecular descriptors like topographical inidices and atom properties**

In [ ]:
from numpy._core import records
def parse_atom_counts(formula: str):
    def _count(symbol):
        m = re.search(rf'{symbol}(\d+)', formula)
        if m:
            return int(m.group(1))
        # Symbol present but no digit → count = 1
        return 1 if re.search(rf'(?<![A-Z]){symbol}(?!\d)', formula) or symbol in formula else 0
    return _count('C'), _count('H'), _count('O')


def compute_rdkit(smiles: str):
    # gives us molecular weight, balaban, hall kier indices, log p values
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, None, None

    mol_h = Chem.AddHs(mol) # adds explicit H for MW
    mw        = round(Descriptors.MolWt(mol_h), 4)
    logp      = round(Descriptors.MolLogP(mol), 4)
    balaban   = round(GraphDescriptors.BalabanJ(mol), 4)
    hall_kier = round(rdMolDescriptors.CalcHallKierAlpha(mol), 4)

    return mw, logp, balaban, hall_kier

def main():
    df = pd.read_excel('/content/smiles_of_species.xlsx')
    print(f"Loaded {len(df)} species.")

    records = []
    failed  = []

    for _, row in df.iterrows():
        species = row['Species']
        formula = str(row['Mol Formula'])
        smiles  = str(row['SMILES'])

        # Atom counts from formula
        n_c, n_h, n_o = parse_atom_counts(formula)

        # RDKit descriptors from SMILES
        mw, logp, balaban, hall = compute_rdkit(smiles)

        if mw is None:
            print(f"  ⚠ Could not parse SMILES for {species}: '{smiles}'")
            failed.append(species)

        records.append({
            'Species'    : species,
            'Mol Formula': formula,
            'SMILES'     : smiles,
            't_n_c'      : n_c,
            't_n_h'      : n_h,
            't_n_o'      : n_o,
            'mw'         : mw,
            'logp'       : logp,
            'balaban'    : balaban,
            'hall'       : hall,
        })

    out = pd.DataFrame(records)

    print("\n===== Descriptor Table =====")
    print(out.to_string(index=False))

    if failed:
        print(f"\n⚠ Failed to compute descriptors for: {failed}")
    out.to_excel('feats_smiles.xlsx')
    return out


if __name__ == '__main__':
    main()

Loaded 46 species.

===== Descriptor Table =====
           Species Mol Formula          SMILES  t_n_c  t_n_h  t_n_o      mw    logp  balaban  hall
              CH2C        C2H2           C=[C]      2      2      0  26.038  0.4824   2.0000 -0.26
             CH2CH        C2H3       [CH][CH2]      2      3      0  27.046  0.5316   1.0000  0.00
            CH2CH2        C2H4             C=C      2      4      0  28.054  0.8022   2.0000 -0.26
           CH2CHCO      C3H3O1        O=[C]C=C      3      3      1  55.056  0.2821   2.7321 -0.59
         CH2CHCOOH      C3H4O2       OC(=O)C=C      3      4      2  72.063  0.2570   3.2048 -0.79
              CH3C        C2H3            C[C]      2      3      0  27.046  0.5944   1.0000  0.00
            CH3CCO      C3H3O1        C[C]=C=O      3      3      1  55.056  0.1973   3.1086 -0.55
           CH3CCOO      C3H3O2    C[C]C(=O)[O]      3      3      2  71.055  0.0447   2.8474 -0.53
          CH3CCOOH      C3H4O2      OC(=O)[C]C      3      4

[13:04:18] WARNING: not removing hydrogen atom without neighbors
[13:04:18] WARNING: not removing hydrogen atom without neighbors


**For every reaction, with the help of SMILES evaluated, we calculate the properties of the reactant side only**

In [ ]:
import re
import pandas as pd
master  = pd.read_excel('/content/Final_Master_cleaned (2).xlsx')
desc_df = pd.read_excel('/content/feats_smiles.xlsx')

desc = desc_df.set_index('Species').to_dict(orient='index')

print(f"Master reactions : {len(master)}")
print(f"Species in lookup: {len(desc)}")

# Parser
def parse_reactant_side(rxn_str):
    """
    Returns:
        species_list  : list of clean species names (stars stripped)
        total_stars   : total star count on reactant side (species stars + bare stars)
    """
    left = rxn_str.split('→')[0].strip()
    parts = [p.strip() for p in left.split('+')]

    species_list = []
    total_stars  = 0

    for part in parts:
        # bare star token: '*', '2*', '3*' etc.
        if re.fullmatch(r'\d*\*+', part):
            total_stars += part.count('*')
        else:
            # count stars attached to species name
            total_stars += part.count('*')
            clean = re.sub(r'\*+$', '', part).strip()
            if clean:
                species_list.append(clean)

    return species_list, total_stars


# Feature computation
SUM_COLS = ['t_n_c', 't_n_h', 't_n_o', 'mw', 'logp', 'balaban', 'hall']

rows = []
missing_log = []

for idx, row in master.iterrows():
    rxn   = row['reaction']
    sp_list, n_stars = parse_reactant_side(rxn)

    totals = {c: 0.0 for c in SUM_COLS}
    row_missing = []

    for sp in sp_list:
        if sp in desc:
            d = desc[sp]
            for c in SUM_COLS:
                val = d.get(c, 0)
                totals[c] += val if pd.notna(val) else 0
        else:
            row_missing.append(sp)

    if row_missing:
        missing_log.append((idx, rxn, row_missing))

    out = {
        'total_stars' : n_stars,
        't_n_c'       : int(totals['t_n_c']),
        't_n_h'       : int(totals['t_n_h']),
        't_n_o'       : int(totals['t_n_o']),
        'sum_mw'      : round(totals['mw'],      4),
        'sum_logp'    : round(totals['logp'],    4),
        'sum_balaban' : round(totals['balaban'], 4),
        'sum_hall'    : round(totals['hall'],    4),
    }
    rows.append(out)

feat_df = pd.DataFrame(rows)

# Merge with master dataset
result = pd.concat([master.reset_index(drop=True), feat_df], axis=1)
result = result.rename(columns={
    'catalyst'   : 'Catalyst',
    'reaction'   : 'Reaction',
    'direction'  : 'Direction',
    'Pauling_EN' : 'pauling_electronegativity',
    'Allen_EN'   : 'allen_en',
})
preview_cols = ['Catalyst','Reaction','Direction','bond_type',
                'total_stars','t_n_c','t_n_h','t_n_o',
                'sum_mw','sum_logp','sum_balaban','sum_hall']
print(result[preview_cols].head(10).to_string(index=False))

Master reactions : 558
Species in lookup: 46
Catalyst                             Reaction Direction       bond_type  total_stars  t_n_c  t_n_h  t_n_o  sum_mw  sum_logp  sum_balaban  sum_hall
      Cu CH3CH2COOH* + 3* → CH3CH2CO*** + OH*   Forward      OH_removal            2      3      6      2  74.079    0.4810       2.8474     -0.53
      Cu CH3CH2COOH* + 3* → CH3CHCOOH*** + H*   Forward dehydrogenation            2      3      6      2  74.079    0.4810       2.8474     -0.53
      Cu      CH3CH2CO*** → CH3CH2* + CO* + *   Forward decarbonylation            3      3      5      1  57.072    0.5061       2.2968     -0.33
      Cu    CH3CH2CO*** + * → CH3CHCO*** + H*   Forward dehydrogenation            4      3      5      1  57.072    0.5061       2.2968     -0.33
      Cu  CH3CHCOOH*** + * → CH3CHCO*** + OH*   Forward      OH_removal            4      3      5      2  73.071    0.2952       2.8474     -0.53
      Cu   CH3CHCOOH*** → CH2CHCOOH* + H* + *   Forward dehydrogenation  

**Computing the molecular fingerprints using rdkit to get bond level features**

In [ ]:
BOND_COLS = [
    'O0', 'O1', 'C0', 'C1', 'C2', 'C3',
    'C-H', 'C-O0', 'C-O1', 'C=O', 'O-H',
    'C0-C0', 'C0-C1', 'C0-C2', 'C0-C3', 'C1-C1', 'C1-C2'
]

INPUT_FILE  = '/content/smiles_of_species.xlsx'
OUTPUT_FILE = 'molecular_fingerprints_computed.xlsx'

def get_atom_type(atom) -> str | None:
    """
    Returns atom type label ('C0'–'C3' or 'O0'/'O1') based on free valence.
    Returns None for atoms other than C and O.
    """
    sym = atom.GetSymbol()
    if sym not in ('C', 'O'):
        return None

    h          = atom.GetNumExplicitHs() + atom.GetNumImplicitHs()
    bo_sum     = sum(b.GetBondTypeAsDouble() for b in atom.GetBonds())
    normal_val = {'C': 4, 'O': 2}[sym]
    free_val   = round(normal_val - bo_sum - h)

    return f'{sym}{free_val}'

def compute_fingerprint(smiles: str) -> dict | None:
    """
    Compute the 17-feature fingerprint for a given SMILES string.
    Returns None if the SMILES cannot be parsed.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp         = {k: 0 for k in BOND_COLS}
    atom_types = {}

    # Atom counts and X–H bond counts
    for atom in mol.GetAtoms():
        t   = get_atom_type(atom)
        sym = atom.GetSymbol()
        h   = atom.GetNumExplicitHs() + atom.GetNumImplicitHs()

        if t is not None:
            atom_types[atom.GetIdx()] = t
            fp[t] += 1

        if sym == 'C' and h > 0:
            fp['C-H'] += h
        if sym == 'O' and h > 0:
            fp['O-H'] += h

    # Heavy-atom bond counts
    for bond in mol.GetBonds():
        i  = bond.GetBeginAtomIdx()
        j  = bond.GetEndAtomIdx()
        ti = atom_types.get(i)
        tj = atom_types.get(j)
        bo = bond.GetBondTypeAsDouble()

        if ti is None or tj is None:
            continue

        si, sj = ti[0], tj[0]

        # C–O and C=O
        if {si, sj} == {'C', 'O'}:
            o_type = ti if si == 'O' else tj
            if bo == 2.0:
                fp['C=O'] += 1
            elif bo == 1.0:
                fp[f'C-{o_type}'] += 1

        # C–C: each bond counted as 1 regardless of bond order
        elif si == 'C' and sj == 'C':
            n1, n2 = int(ti[1]), int(tj[1])
            key    = f'C{min(n1, n2)}-C{max(n1, n2)}'
            if key in fp:
                fp[key] += 1

    return fp

# Main
def main():
    df = pd.read_excel(INPUT_FILE)
    print(f"Loaded {len(df)} species from '{INPUT_FILE}'")

    records, failed = [], []

    for _, row in df.iterrows():
        species = row['Species']
        smiles  = str(row['SMILES'])
        fp      = compute_fingerprint(smiles)

        if fp is None:
            print(f"  ⚠ Invalid SMILES for {species}: '{smiles}'")
            failed.append(species)
            fp = {k: None for k in BOND_COLS}

        records.append({
            'Species'    : species,
            'Mol Formula': row['Mol Formula'],
            'SMILES'     : smiles,
            **fp
        })

    out = pd.DataFrame(records)
    out.to_excel(OUTPUT_FILE, index=False)
    return out

if __name__ == '__main__':
    main()

Loaded 46 species from '/content/smiles_of_species.xlsx'


[13:04:19] WARNING: not removing hydrogen atom without neighbors
[13:04:19] WARNING: not removing hydrogen atom without neighbors


**For every reaction, we calculate the molecular fingerprints of the reactant side only**

In [ ]:
MASTER_FILE = '/content/Final_Master_cleaned (2).xlsx'
FP_FILE     = '/content/molecular_fingerprints_computed.xlsx'
OUT_FILE    = 'master_with_fingerprints.xlsx'

FP_COLS = [
    'O0', 'O1', 'C0', 'C1', 'C2', 'C3',
    'C-H', 'C-O0', 'C-O1', 'C=O', 'O-H',
    'C0-C0', 'C0-C1', 'C0-C2', 'C0-C3', 'C1-C1', 'C1-C2'
]

REACTANT_COLS = [f'sum_reactant_{c}' for c in FP_COLS]
PRODUCT_COLS  = [f'sum_product_{c}'  for c in FP_COLS]

#  Helpers function
def parse_side(side_str: str) -> list[str]:
    """
    Extract clean species names from one side of a reaction string.
    Strips trailing stars and ignores bare star tokens (e.g. '3*', '*').
    """
    species = []
    for part in side_str.split('+'):
        part = part.strip()
        if re.fullmatch(r'\d*\*+', part):
            continue                          # bare star token — skip
        clean = re.sub(r'\*+$', '', part).strip()
        if clean:
            species.append(clean)
    return species

def sum_fingerprints(species_list: list[str], lookup: dict) -> dict:
    #Sum fingerprint values across a list of species.
    totals = {c: 0 for c in FP_COLS}
    for sp in species_list:
        if sp in lookup:
            for c in FP_COLS:
                v = lookup[sp].get(c, 0)
                totals[c] += v if v is not None else 0
        else:
            print(f"  ⚠ Species '{sp}' not found in fingerprint lookup")
    return totals

def main():
    master = pd.read_excel(MASTER_FILE)
    fp_df  = pd.read_excel(FP_FILE)
    lookup = fp_df.set_index('Species')[FP_COLS].to_dict(orient='index')
    print(f"Loaded {len(master)} reactions | {len(lookup)} species in fingerprint lookup")
    reactant_rows = []
    product_rows  = []

    for idx, row in master.iterrows():
        rxn = row['reaction']
        left, right = rxn.split('→')

        r_species = parse_side(left)
        p_species = parse_side(right)

        r_fp = sum_fingerprints(r_species, lookup)
        p_fp = sum_fingerprints(p_species, lookup)

        reactant_rows.append({f'sum_reactant_{c}': v for c, v in r_fp.items()})
        product_rows.append( {f'sum_product_{c}':  v for c, v in p_fp.items()})

    r_df = pd.DataFrame(reactant_rows)
    p_df = pd.DataFrame(product_rows)
    result = pd.concat([
        master.reset_index(drop=True),
        r_df.reset_index(drop=True),
        p_df.reset_index(drop=True)
    ], axis=1)
    result = result.rename(columns={
    'catalyst'   : 'Catalyst',
    'reaction'   : 'Reaction',
    'direction'  : 'Direction',
    'Pauling_EN' : 'pauling_electronegativity',
    'Allen_EN'   : 'allen_en',
})

    print("\n===== Sample output (first 5 rows, fingerprint columns) =====")
    preview = ['Catalyst', 'Reaction'] + REACTANT_COLS[:6] + PRODUCT_COLS[:6]
    print(result[preview].head(5).to_string(index=False))
    result.to_excel(OUT_FILE, index=False)
    print(f"\nSaved → {OUT_FILE}  ({result.shape[0]} rows × {result.shape[1]} cols)")
    print(f"New columns added: {REACTANT_COLS + PRODUCT_COLS}")

    return result


if __name__ == '__main__':
    main()

Loaded 558 reactions | 46 species in fingerprint lookup

===== Sample output (first 5 rows, fingerprint columns) =====
Catalyst                             Reaction  sum_reactant_O0  sum_reactant_O1  sum_reactant_C0  sum_reactant_C1  sum_reactant_C2  sum_reactant_C3  sum_product_O0  sum_product_O1  sum_product_C0  sum_product_C1  sum_product_C2  sum_product_C3
      Cu CH3CH2COOH* + 3* → CH3CH2CO*** + OH*                2                0                3                0                0                0               1               1               2               1               0               0
      Cu CH3CH2COOH* + 3* → CH3CHCOOH*** + H*                2                0                3                0                0                0               2               0               2               1               0               0
      Cu      CH3CH2CO*** → CH3CH2* + CO* + *                1                0                2                1                0                0  